In [ ]:
import os
os.environ["TF_CPP_MIN_LOG_LEVEL"] = "3"      # hide TensorFlow's start-up messages
from pathlib import Path
import numpy as np
import pandas as pd
from PIL import Image
import keras
import tensorflow as tf

keras.utils.set_random_seed(42)
tf.config.experimental.enable_op_determinism()
DATA = Path("data")

## How much memory one convolution layer's output takes

In [ ]:
n, f, k = 224, 3, 100                       # 224 x 224 x 3 image, 100 filters of 3 x 3
side = n - f + 1
values = side * side * k
print(f"feature maps: {side} x {side} x {k} = {values:,} numbers")
print(f"one image, 32-bit floats: {values * 4 / 1e6:.1f} MB")
print(f"a batch of 100 images:    {values * 4 * 100 / 1e9:.2f} GB")
p = side // 2
print(f"after 2x2 max pooling: {p} x {p} x {k}, {p * p * k * 4 / 1e6:.1f} MB per image")

## Max pooling by hand: 2 x 2 window, stride 2

In [ ]:
A = np.array([[1, 5, 2, 3], [2, 4, 0, 1], [7, 1, 4, 2], [3, 0, 1, 3]], float)

def pool(A, size=2, stride=2, op=np.max):
    n_out = (A.shape[0] - size) // stride + 1
    return np.array([[op(A[i * stride:i * stride + size, j * stride:j * stride + size])
                      for j in range(n_out)] for i in range(n_out)])

print("max pooling:\n", pool(A))
print("average pooling:\n", pool(A, op=np.mean))
print("L2 pooling:\n", pool(A, op=lambda w: np.sqrt((w ** 2).sum())).round(2))
print("Keras MaxPooling2D:\n", keras.layers.MaxPooling2D(2, 2)(A.reshape(1, 4, 4, 1)).numpy()[0, :, :, 0])
print("Keras AveragePooling2D:\n", keras.layers.AveragePooling2D(2, 2)(A.reshape(1, 4, 4, 1)).numpy()[0, :, :, 0])
print("global max:", keras.layers.GlobalMaxPooling2D()(A.reshape(1, 4, 4, 1)).numpy(),
      " global average:", keras.layers.GlobalAveragePooling2D()(A.reshape(1, 4, 4, 1)).numpy())

## Pooling a volume: each feature map on its own

In [ ]:
V = np.random.default_rng(0).random((1, 4, 4, 2))
print("input", V.shape, "-> max pooled", keras.layers.MaxPooling2D(2)(V).shape,
      "-> global max", keras.layers.GlobalMaxPooling2D()(V).shape)
print("first map pooled alone equals channel 0:",
      np.allclose(pool(V[0, :, :, 0]), keras.layers.MaxPooling2D(2)(V).numpy()[0, :, :, 0]))

## Pooling layers in a Keras model: shapes and parameters

In [ ]:
model = keras.Sequential([
    keras.Input(shape=(28, 28, 1)),
    keras.layers.Conv2D(32, 3, activation="relu"),
    keras.layers.MaxPooling2D(pool_size=(2, 2), strides=2),
    keras.layers.Conv2D(32, 3, activation="relu"),
    keras.layers.MaxPooling2D(pool_size=(2, 2), strides=2),
    keras.layers.Flatten(),
    keras.layers.Dense(10, activation="softmax")])
for layer in model.layers:
    print(f"{layer.name:16s} {str(layer.output.shape):20s} {layer.count_params():6d} parameters")

## A shifted digit: the feature map moves, the pooled map barely changes
A vertical-edge filter and ReLU on 1000 MNIST test digits, each also shifted 1 pixel to the right.
We measure how much each representation changes: |after shift - before| / |before|.

In [ ]:
(_, _), (x_test, y_test) = keras.datasets.mnist.load_data()
X = x_test[:1000, :, :, None].astype("float32") / 255.0
Xs = np.roll(X, 1, axis=2)                                   # move every digit 1 pixel to the right
K = np.array([[1, 0, -1]] * 3, "float32").reshape(3, 3, 1, 1)
conv = lambda x: tf.nn.relu(tf.nn.conv2d(x, K, 1, "VALID")).numpy()
F, Fs = conv(X), conv(Xs)
print("feature map moves with the input:", np.allclose(Fs[:, :, 1:], F[:, :, :-1]))

def change(a, b):
    a, b = a.reshape(len(a), -1), b.reshape(len(b), -1)
    return float(np.mean(np.linalg.norm(a - b, axis=1) / np.linalg.norm(a, axis=1)))

rows = [dict(stage="feature map (no pooling)", change=change(F, Fs))]
for size in (2, 3, 4):
    mp = keras.layers.MaxPooling2D(size)
    rows.append(dict(stage=f"after {size}x{size} max pooling", change=change(mp(F).numpy(), mp(Fs).numpy())))
gm = keras.layers.GlobalMaxPooling2D()
rows.append(dict(stage="after global max pooling", change=change(gm(F).numpy(), gm(Fs).numpy())))
shift = pd.DataFrame(rows)
print(shift.round(3).to_string(index=False))
shift.to_csv(DATA / "shift_change.csv", index=False)
d = 3                                                         # one digit for the figure
np.savez_compressed(DATA / "shift_example.npz", x=X[d, :, :, 0], xs=Xs[d, :, :, 0], f=F[d, :, :, 0],
                    fs=Fs[d, :, :, 0], p=keras.layers.MaxPooling2D(2)(F[d:d + 1]).numpy()[0, :, :, 0],
                    ps=keras.layers.MaxPooling2D(2)(Fs[d:d + 1]).numpy()[0, :, :, 0])
print("digit", y_test[d])

## Max against average pooling on a photo's edge map

In [ ]:
photo = np.array(Image.open(DATA / "photo.png"), dtype="float32")[None, :, :, None]
E = tf.nn.relu(tf.nn.conv2d(photo, K, 1, "VALID")).numpy()     # bright-left edges of the photo
Emax = keras.layers.MaxPooling2D(4)(E).numpy()
Eavg = keras.layers.AveragePooling2D(4)(E).numpy()
print("edge map", E.shape[1:3], "-> pooled", Emax.shape[1:3])
print("mean value: edge map %.1f, max pooled %.1f, average pooled %.1f" % (E.mean(), Emax.mean(), Eavg.mean()))
top = np.percentile(E, 99)
print("share of pooled cells above the edge map's 99th percentile: max %.3f, average %.3f"
      % ((Emax > top).mean(), (Eavg > top).mean()))
np.savez_compressed(DATA / "photo_pool.npz", E=E[0, :, :, 0], Emax=Emax[0, :, :, 0], Eavg=Eavg[0, :, :, 0])

## Global average pooling instead of Flatten: parameters of the last layer

In [ ]:
for name, head in (("Flatten", keras.layers.Flatten()), ("GlobalAveragePooling2D", keras.layers.GlobalAveragePooling2D())):
    m = keras.Sequential([keras.Input(shape=(5, 5, 32)), head, keras.layers.Dense(10)])
    print(f"{name:24s} -> {m.layers[0].output.shape[1]:4d} values, Dense(10) has {m.count_params():5d} parameters")

## Shifting further: 0 to 3 pixels
The same 1000 digits shifted 0, 1, 2 and 3 pixels to the right. Pooling gives invariance to small shifts only: the change grows with the shift, and a window of 2 cannot hide a shift of 2 or 3.

In [ ]:
sweep = []
for s in range(4):
    Fk = conv(np.roll(X, s, axis=2))
    row = dict(shift=s, no_pooling=change(F, Fk))
    for size in (2, 4):
        mp = keras.layers.MaxPooling2D(size)
        row[f"max_{size}x{size}"] = change(mp(F).numpy(), mp(Fk).numpy())
    row["global_max"] = change(gm(F).numpy(), gm(Fk).numpy())
    sweep.append(row)
sweep = pd.DataFrame(sweep)
print(sweep.round(3).to_string(index=False))
sweep.round(5).to_csv(DATA / "shift_sweep.csv", index=False)
d8 = int(np.argmax(y_test[:1000] == 8))                       # the first 8 of the test set, for the animation
x8 = np.stack([np.roll(X[d8], s, axis=1) for s in range(4)])  # the same 8 shifted 0..3 pixels
f8 = conv(x8)
np.savez_compressed(DATA / "shift_frames.npz", x=x8[..., 0], f=f8[..., 0],
                    p=keras.layers.MaxPooling2D(2)(f8).numpy()[..., 0])
print("digit index", d8, "label", y_test[d8])